In [45]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [46]:
df=pd.read_csv(r"C:\Users\ADMIN\OneDrive\ドキュメント\ML\project2\crop_yield_database.csv")

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Record_ID                 3000 non-null   int64  
 1   Crop                      3000 non-null   object 
 2   State                     3000 non-null   object 
 3   Season                    3000 non-null   object 
 4   Area_hectares             3000 non-null   float64
 5   Annual_Rainfall_mm        3000 non-null   float64
 6   Avg_Temperature_C         3000 non-null   float64
 7   Humidity_percent          3000 non-null   float64
 8   Soil_Type                 3000 non-null   object 
 9   Soil_pH                   3000 non-null   float64
 10  N_kg_per_ha               3000 non-null   float64
 11  P_kg_per_ha               3000 non-null   float64
 12  K_kg_per_ha               3000 non-null   float64
 13  Fertilizer_kg_per_ha      3000 non-null   float64
 14  Irrigati

In [48]:
df.head()

,Record_ID,Crop,State,Season,Area_hectares,Annual_Rainfall_mm,Avg_Temperature_C,Humidity_percent,Soil_Type,Soil_pH,N_kg_per_ha,P_kg_per_ha,K_kg_per_ha,Fertilizer_kg_per_ha,Irrigation_Method,Pesticide_kg_per_ha,Yield_tonnes_per_hectare,Total_Production_tonnes
0,1,Rice,Tamil Nadu,Rabi,42.97,1179.2,18.2,42.5,Black,6.60,75.5,41.7,27.9,149.6,Sprinkler,6.14,3.594,154.420
1,2,Sugarcane,Uttar Pradesh,Annual,3.42,1157.5,27.8,59.2,Alluvial,5.96,121.7,39.2,36.4,94.2,Canal,7.65,81.857,279.950
2,3,Soybean,Andhra Pradesh,Kharif,48.38,670.7,22.3,71.2,Alluvial,7.40,81.6,26.9,28.5,139.3,Canal,8.65,1.516,73.321
3,4,Pulses,Madhya Pradesh,Rabi,34.20,928.3,27.1,63.0,Black,6.73,103.9,15.8,38.6,88.8,Borewell,4.68,0.819,28.008
4,5,Millet,Karnataka,Rabi,38.31,907.0,30.6,72.7,Black,7.13,74.5,33.7,62.2,101.4,Drip,2.06,1.332,51.026


In [49]:
df.isnull().sum()

Record_ID                   0
Crop                        0
State                       0
Season                      0
Area_hectares               0
Annual_Rainfall_mm          0
Avg_Temperature_C           0
Humidity_percent            0
Soil_Type                   0
Soil_pH                     0
N_kg_per_ha                 0
P_kg_per_ha                 0
K_kg_per_ha                 0
Fertilizer_kg_per_ha        0
Irrigation_Method           0
Pesticide_kg_per_ha         0
Yield_tonnes_per_hectare    0
Total_Production_tonnes     0
dtype: int64

In [50]:
df.duplicated().sum()

np.int64(0)

In [51]:
df.shape

(3000, 18)

In [55]:
df.drop(["Record_ID", "Total_Production_tonnes", "Irrigation_Method", "Pesticide_kg_per_ha"],axis=1,inplace=True)

In [56]:
df.head()

,Crop,State,Season,Area_hectares,Annual_Rainfall_mm,Avg_Temperature_C,Humidity_percent,Soil_Type,Soil_pH,N_kg_per_ha,P_kg_per_ha,K_kg_per_ha,Fertilizer_kg_per_ha,Yield_tonnes_per_hectare
0,Rice,Tamil Nadu,Rabi,42.97,1179.2,18.2,42.5,Black,6.60,75.5,41.7,27.9,149.6,3.594
1,Sugarcane,Uttar Pradesh,Annual,3.42,1157.5,27.8,59.2,Alluvial,5.96,121.7,39.2,36.4,94.2,81.857
2,Soybean,Andhra Pradesh,Kharif,48.38,670.7,22.3,71.2,Alluvial,7.40,81.6,26.9,28.5,139.3,1.516
3,Pulses,Madhya Pradesh,Rabi,34.20,928.3,27.1,63.0,Black,6.73,103.9,15.8,38.6,88.8,0.819
4,Millet,Karnataka,Rabi,38.31,907.0,30.6,72.7,Black,7.13,74.5,33.7,62.2,101.4,1.332


In [57]:
print("Dataset shape:", df.shape)

Dataset shape: (3000, 14)


In [58]:
target_col = "Yield_tonnes_per_hectare"

In [59]:
X = df.drop(columns=[target_col])
y = df[target_col]

In [60]:
cat_fea = X.select_dtypes(include=["object"]).columns.tolist()
num_fea = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:", cat_fea)
print("Numerical features:", num_fea)

Categorical features: ['Crop', 'State', 'Season', 'Soil_Type']
Numerical features: ['Area_hectares', 'Annual_Rainfall_mm', 'Avg_Temperature_C', 'Humidity_percent', 'Soil_pH', 'N_kg_per_ha', 'P_kg_per_ha', 'K_kg_per_ha', 'Fertilizer_kg_per_ha']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [62]:
# 5. PREPROCESSOR PIPELINE
# ==========================================
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_fea),
        ("num", StandardScaler(), num_fea)
    ]
)

In [63]:
# 6. MODEL PIPELINE (XGBOOST)
# ==========================================
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    ))
])

In [64]:
# Train pipeline directly on raw features
model_pipeline.fit(X_train, y_train)
print("Crop Yield pipeline trained successfully!")

Crop Yield pipeline trained successfully!


In [65]:
y_pred = model_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("-" * 40)
print("EVALUATION METRICS (TEST SET)")
print("-" * 40)
print(f"MAE:      {mae:.4f} tonnes/ha")
print(f"MSE:      {mse:.4f}")
print(f"RMSE:     {rmse:.4f} tonnes/ha")
print(f"R² Score: {r2:.4f}")
print("-" * 40)

----------------------------------------
EVALUATION METRICS (TEST SET)
----------------------------------------
MAE:      0.9922 tonnes/ha
MSE:      7.2881
RMSE:     2.6996 tonnes/ha
R² Score: 0.9796
----------------------------------------


In [67]:
import joblib

In [68]:
model_filename = "crop_yield_model.pkl"
joblib.dump(model_pipeline, model_filename)
print(f"Model successfully saved to {model_filename}")

Model successfully saved to crop_yield_model.pkl


In [69]:
loaded_pipeline = joblib.load(model_filename)
print("Model loaded successfully!")

Model loaded successfully!
